# Portal Tugas Akhir Trunojoyo

In [1]:
!pip install builtwith


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import requests
from bs4 import BeautifulSoup
import csv
import time

In [3]:
BASE_URL = "https://pta.trunojoyo.ac.id"

In [4]:
def get_prodi_links():
    resp = requests.get(BASE_URL)
    if resp.status_code != 200:
        print(f"Gagal akses {BASE_URL} ({resp.status_code})")
        return []

    soup = BeautifulSoup(resp.text, "html.parser")

    # Sidebar “Journal” ada di halaman utama
    sidebar = soup.find("div", class_="sidebar_nav")
    if not sidebar:
        print("[Sidebar tidak ditemukan]")
        return []

    prodi_links = []
    # Loop tiap fakultas (li tingkat atas)
    for f_li in sidebar.select("ul > li"):
        fakultas = f_li.find("a").get_text(strip=True)
        for p_li in f_li.select("ul > li"):
            prodi = p_li.find("a").get_text(strip=True)
            link = p_li.find("a")["href"]
            print(f"[{fakultas}] {prodi} -> {link}")
            prodi_links.append({"fakultas": fakultas, "prodi": prodi, "url": link})

    print(f"\nTotal prodi: {len(prodi_links)}")
    return prodi_links

if __name__ == "__main__":
    daftar_prodi = get_prodi_links()


[Hukum] Ilmu Hukum -> https://pta.trunojoyo.ac.id/c_search/byprod/1
[Hukum] Magister Ilmu Hukum -> https://pta.trunojoyo.ac.id/c_search/byprod/24
[Pertanian] Teknologi Industri Pertanian -> https://pta.trunojoyo.ac.id/c_search/byprod/2
[Pertanian] Agribisnis -> https://pta.trunojoyo.ac.id/c_search/byprod/3
[Pertanian] Agroteknologi -> https://pta.trunojoyo.ac.id/c_search/byprod/4
[Pertanian] Ilmu Kelautan -> https://pta.trunojoyo.ac.id/c_search/byprod/5
[Pertanian] Manajemen Sumberdaya Perairan -> https://pta.trunojoyo.ac.id/c_search/byprod/35
[Pertanian] Magister Pengelolaan Sumber Daya Alam -> https://pta.trunojoyo.ac.id/c_search/byprod/37
[Ekonomi Dan Bisnis] Ekonomi Pembangunan -> https://pta.trunojoyo.ac.id/c_search/byprod/6
[Ekonomi Dan Bisnis] Manajemen -> https://pta.trunojoyo.ac.id/c_search/byprod/7
[Ekonomi Dan Bisnis] Akuntansi -> https://pta.trunojoyo.ac.id/c_search/byprod/8
[Ekonomi Dan Bisnis] D3 Akuntansi -> https://pta.trunojoyo.ac.id/c_search/byprod/21
[Ekonomi Dan Bis

In [5]:
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://pta.trunojoyo.ac.id/c_search/byprod/1"

# --- fungsi ambil semua link "Selengkapnya" dari 1 halaman prodi ---
def get_detail_links(url):
    soup = BeautifulSoup(requests.get(url).text, "html.parser")
    content = soup.find("div", id="content_journal")
    if not content:
        return []
    
    links = []
    for li in content.find_all("li"):
        a_selengkapnya = li.find("a", class_="gray button")
        if a_selengkapnya and "href" in a_selengkapnya.attrs:
            links.append(a_selengkapnya["href"])
    return links

# --- fungsi ambil judul + abstrak bahasa Inggris dari halaman detail ---
def get_journal_detail(detail_url):
    soup = BeautifulSoup(requests.get(detail_url).text, "html.parser")
    content = soup.find("div", id="content_journal")
    if not content:
        return None
    
    # ambil judul
    title_tag = content.find("a", class_="title")
    title = title_tag.get_text(strip=True) if title_tag else "N/A"
    
    # ambil abstrak bahasa Inggris
    abstrak_en = ""
    for b in content.find_all("b"):
        if "Abstraction" in b.get_text():
            p_tag = b.find_next("p")
            if p_tag:
                abstrak_en = p_tag.get_text(" ", strip=True)
            break
    
    return {"title": title, "abstract_en": abstrak_en}

# --- main scraper ---
if __name__ == "__main__":
    PRODI_ID = 1   # Ilmu Hukum
    MAX_PAGE = 3   # contoh dulu ambil 3 halaman pertama
    
    all_results = []
    for page in range(1, MAX_PAGE+1):
        url_page = f"{BASE_URL}/c_search/byprod/{PRODI_ID}/{page}"
        print(f"[INFO] Scraping halaman {page}: {url_page}")
        
        detail_links = get_detail_links(url_page)
        for link in detail_links:
            detail = get_journal_detail(link)
            if detail:
                all_results.append(detail)
                print("Judul:", detail["title"])
                print("Abstract (EN):", detail["abstract_en"][:500], "...\n")
    
    print(f"\nTotal data terkumpul: {len(all_results)}")


[INFO] Scraping halaman 1: https://pta.trunojoyo.ac.id/c_search/byprod/1/c_search/byprod/1/1
Judul: Implementasi Fungsi Legislasi Dewan Perwakilan Rakyat Daerah Kabupaten Bangkalan Periode 2009-2014 Dalam Pembentukan Peraturan Daerah
Abstract (EN): ABSTRACT
       Implementation of Legislation Parliament function Bangkalan period 2009-2014 in accordance with local regulation Formation of the Constitution of the Republic of Indonesia Year 1945 and Law of the Republic of Indonesia Number 12 Year 2011 On Establishment of Legislation and Law of the Republic of Indonesia Number 32 Year 2004 on Regional Government determines that a legislative function in the hands of Parliament. Legislation functions held by Parliament is a function of the est ...

Judul: Pertanggungjawaban Pidana Direksi BUMN (Persero)
(Anotasi Putusan No. 1144 K/Pid/2006)
Abstract (EN): State Owned Enterprises (SOEs) are business entities that are partly or wholly owned by the State from the wealth separated state. Howeve

Judul: Implementasi Fungsi Legislasi Dewan Perwakilan Rakyat Daerah Kabupaten Bangkalan Periode 2009-2014 Dalam Pembentukan Peraturan Daerah
Abstract (EN): ABSTRACT
       Implementation of Legislation Parliament function Bangkalan period 2009-2014 in accordance with local regulation Formation of the Constitution of the Republic of Indonesia Year 1945 and Law of the Republic of Indonesia Number 12 Year 2011 On Establishment of Legislation and Law of the Republic of Indonesia Number 32 Year 2004 on Regional Government determines that a legislative function in the hands of Parliament. Legislation functions held by Parliament is a function of the est ...

Judul: Pertanggungjawaban Pidana Direksi BUMN (Persero)
(Anotasi Putusan No. 1144 K/Pid/2006)
Abstract (EN): State Owned Enterprises (SOEs) are business entities that are partly or wholly owned by the State from the wealth separated state. However, all the provisions applicable to the state (Limited) refers to the Limited Liability Compa

In [14]:
def get_soup(url):
    resp = requests.get(url)
    if resp.status_code == 200:
        return BeautifulSoup(resp.text, "html.parser")
    return None

In [6]:
# --- fungsi ambil judul & abstrak EN per prodi ---
def get_judul_abstract_en(url_prodi):
    hasil = []
    page = 1
    while True:
        page_url = f"{url_prodi}/{page}"
        soup = get_soup(page_url)
        if not soup:
            break

        content = soup.find("div", id="content_journal")
        if not content:
            break

        lis = content.find_all("li")
        if not lis:
            break

        for li in lis:
            judul_tag = li.find("a", class_="title")
            if not judul_tag:
                continue
            judul = judul_tag.get_text(strip=True)

            abstrak_en = ""
            abs_tag = li.find("p", class_="abstract_en")
            if abs_tag:
                abstrak_en = abs_tag.get_text(" ", strip=True)

            hasil.append({"judul": judul, "abstrak_en": abstrak_en})

        page += 1

    return hasil

In [7]:
# --- lanjut dari daftar_prodi yang sudah kamu punya ---
if __name__ == "__main__":
    # misal daftar_prodi sudah ada dari code sebelumnya:
    # daftar_prodi = [{"fakultas": "...", "prodi": "...", "url": "https://pta.trunojoyo.ac.id/c_search/byprod/1"}, ...]

    all_data = []
    count_per_prodi = {}

    for prodi_info in daftar_prodi:   # tinggal lanjut pakai daftar_prodi yang sudah kamu punya
        fakultas = prodi_info["fakultas"]
        prodi = prodi_info["prodi"]
        url = prodi_info["url"]

        print(f"[INFO] Scraping {prodi} ({url}) ...")
        data = get_judul_abstract_en(url)
        count_per_prodi[prodi] = len(data)

        for d in data:
            all_data.append({
                "fakultas": fakultas,
                "prodi": prodi,
                "judul": d["judul"],
                "abstrak_en": d["abstrak_en"]
            })

    # simpan ke CSV
    with open("hasil_ptatrunojoyo.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["fakultas", "prodi", "judul", "abstrak_en"])
        writer.writeheader()
        writer.writerows(all_data)

    # cetak statistik
    print("\n--- Statistik ---")
    total = 0
    for prodi, n in count_per_prodi.items():
        print(f"{prodi}: {n} judul")
        total += n
    print(f"\nTOTAL: {total} judul")


[INFO] Scraping Ilmu Hukum (https://pta.trunojoyo.ac.id/c_search/byprod/1) ...


NameError: name 'get_soup' is not defined